# Letterform Classification Inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ipavlopoulos/diachronic-greek-letterforms/blob/main/Inference.ipynb)

This notebook loads the released CNN checkpoint and predicts the Greek letter class for one available cliplet image. GitHub's notebook preview is static; use Colab or local Jupyter to run the cells.

In [ ]:
import contextlib
import io
import os
import random
import subprocess
import sys
import warnings
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/ipavlopoulos/diachronic-greek-letterforms.git"
REPO_DIR = Path("/content/diachronic-greek-letterforms")

if IN_COLAB and not Path("source.py").exists():
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    import torch
    from IPython.display import display
    from PIL import Image

    from source import LETTER_LABELS, image_path_to_tensor, load_letterform_model

print(f"Ready. Running in {'Colab' if IN_COLAB else 'local Jupyter'} from {Path.cwd()}")

## 1. Select a Cliplet

The model expects a cropped image containing one Greek letterform. This example samples from `data/palitchar/cliplets`; you can switch `cliplet_dir` to `data/medchar/cliplets` if desired.

In [ ]:
cliplet_dir = Path("data/palitchar/cliplets")
image_paths = sorted(
    p for p in cliplet_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
selected_path = random.choice(image_paths)

selected_path, len(image_paths)

In [ ]:
display(Image.open(selected_path).convert("L").resize((160, 160)))

## 2. Load the Model and Predict

The checkpoint predicts one of the 24 Greek letter classes. The image is converted to grayscale, resized to `64x64`, and normalized through the shared preprocessing helper.

In [ ]:
device = "cpu"
model = load_letterform_model("best_cnn_letter_model.pth", device=device)
image_tensor = image_path_to_tensor(selected_path, device=device)

with torch.no_grad():
    logits = model(image_tensor)
    probabilities = torch.softmax(logits, dim=1)[0]
    predicted_index = int(probabilities.argmax())

print(f"Image: {selected_path}")
print(f"Predicted class: {LETTER_LABELS[predicted_index]}")
print(f"Confidence: {float(probabilities[predicted_index]):.3f}")